# Forex Strategy Optimizer

This notebook provides a Meta Trader-style strategy tester that exhaustively tests all possible combinations of filters.

## How it works

1. **Filter Dimensions**: Each column from the CSV (EMA, BOS/CH, SL, News, Hour, etc.) becomes a "dimension"
2. **Combinations**: The optimizer generates all possible combinations of these filters
3. **Backtesting**: Each combination is backtested across all RRR ratios (1:1, 1:2, 1:3)
4. **Results**: Strategies are ranked by edge (profitability above breakeven)

## Parameters

- **max_filters**: Maximum number of filters per strategy (e.g., 3 = up to 3 conditions)
- **min_trades**: Minimum trade count required (e.g., 10 trades minimum)
- **min_edge**: Minimum edge percentage required (e.g., 5.0 = 5% above breakeven)
- **top_n**: Show only top N strategies (or None for all)

In [2]:
# Import required libraries
import sys
import importlib

# Add parent directory to path
sys.path.insert(0, '..')

# Enable automatic reloading
%load_ext autoreload
%autoreload 2

# Import utilities
from utils import load_and_clean_data
from utils.optimizer import (
    optimize_strategies,
    display_optimization_results,
    export_optimization_results
)

# Load the data
print("Loading data...")
df = load_and_clean_data()
print(f"Loaded {len(df)} trades")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading data...
Loaded 1102 trades


## Quick Test: Top 50 Strategies (Max 2 Filters)

Start with a quick test to see the best double-setup strategies.

In [6]:
# Quick optimization: max 2 filters, top 50 results
results = optimize_strategies(
    df,
    max_filters=2,         # Test up to 2 filters per strategy
    min_filters=1,         # Include single-filter strategies too
    min_trades=100,        # Require at least 100 trades
    min_edge=0.0,          # Show all strategies (even negative edge)
    rrr_ratios=[1, 2, 3],  # Test all RRR ratios
    top_n=None             # Unlimited list
)

display_optimization_results(results, title="Top 50 Strategies (Max 2 Filters)")

Generating strategy combinations (max 2 filters)...
Total combinations to test: 807

Backtesting 807 strategies...
  Processed 500/807 strategies...

Optimization complete! Found 322 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,SL=5-10 + Hour=10-12,128,1:2,44.5%,11.2%,1.61,0.34R,55.5%
2,SL=5-10 + Hour=10-12,128,1:3,32.8%,7.8%,1.47,0.31R,67.2%
3,EMA=Aligned + Weekday=Tuesday,140,1:2,40.0%,6.7%,1.33,0.20R,60.0%
4,SL=5-10 + News=News >2h,123,1:2,39.8%,6.5%,1.32,0.20R,60.2%
5,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
6,Hour=10-12 + Direction=Buy,153,1:2,39.2%,5.9%,1.29,0.18R,60.8%
7,Hour=15,117,1:3,30.8%,5.8%,1.33,0.23R,69.2%
8,Hour=15,117,1:3,30.8%,5.8%,1.33,0.23R,69.2%
9,Hour=15,117,1:3,30.8%,5.8%,1.33,0.23R,69.2%
10,Hour=15,117,1:3,30.8%,5.8%,1.33,0.23R,69.2%


## Full Optimization: All 3-Filter Combinations

Run a comprehensive test with up to 3 filters per strategy. This will test thousands of combinations.

In [8]:
# Full optimization: max 3 filters, top 100 results
results = optimize_strategies(
    df,
    max_filters=3,         # Test up to 3 filters per strategy
    min_filters=1,         # Include single-filter strategies
    min_trades=200,        # Require at least 100 trades
    min_edge=5.0,          # Only show strategies with 5%+ edge
    rrr_ratios=[1, 2, 3],  # Test all RRR ratios
    top_n=None             # Unlimited
)

display_optimization_results(results, title="Top 100 Strategies (Max 3 Filters, 5%+ Edge)")

Generating strategy combinations (max 3 filters)...
Total combinations to test: 8617

Backtesting 8617 strategies...
  Processed 500/8617 strategies...
  Processed 1000/8617 strategies...
  Processed 1500/8617 strategies...
  Processed 2000/8617 strategies...
  Processed 2500/8617 strategies...
  Processed 3000/8617 strategies...
  Processed 3500/8617 strategies...
  Processed 4000/8617 strategies...
  Processed 4500/8617 strategies...
  Processed 5000/8617 strategies...
  Processed 5500/8617 strategies...
  Processed 6000/8617 strategies...
  Processed 6500/8617 strategies...
  Processed 7000/8617 strategies...
  Processed 7500/8617 strategies...
  Processed 8000/8617 strategies...
  Processed 8500/8617 strategies...

Optimization complete! Found 8 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
2,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
3,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
4,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
5,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
6,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
7,SL=5-10 + Direction=Buy,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
8,EMA=Aligned + BOS/CH=BOS + SL=5-10,227,1:2,38.8%,5.5%,1.27,0.16R,61.2%


## Aggressive Optimization: 4+ Filters

Test complex strategies with 4+ filters. Warning: This can take longer and may overfit to the data.

In [ ]:
# Aggressive optimization: max 4 filters
results = optimize_strategies(
    df,
    max_filters=4,      # Test up to 4 filters per strategy
    min_filters=3,      # Only show 3+ filter strategies
    min_trades=5,       # Lower minimum (complex filters = fewer trades)
    min_edge=10.0,      # High edge threshold (10%+)
    rrr_ratios=[1, 2, 3],
    top_n=50
)

display_optimization_results(results, title="Top 50 Complex Strategies (3-4 Filters, 10%+ Edge)")

## Custom Optimization

Customize your own optimization parameters below:

In [12]:
# Custom optimization - adjust parameters as needed
custom_results = optimize_strategies(
    df,
    max_filters=3,           # Your choice: 1, 2, 3, 4, etc.
    min_filters=1,           # Your choice: 1, 2, 3, etc.
    min_trades=600,           # Your choice: adjust based on data size
    min_edge=0.0,            # Your choice: 0 for all, 5 for 5%+, etc.
    rrr_ratios=[3],    # Your choice: [1], [2], [3], or [1,2,3]
    top_n=None               # Your choice: None for all, or a number
)

display_optimization_results(custom_results, title="Custom Optimization Results")

Generating strategy combinations (max 3 filters)...
Total combinations to test: 8617

Backtesting 8617 strategies...
  Processed 500/8617 strategies...
  Processed 1000/8617 strategies...
  Processed 1500/8617 strategies...
  Processed 2000/8617 strategies...
  Processed 2500/8617 strategies...
  Processed 3000/8617 strategies...
  Processed 3500/8617 strategies...
  Processed 4000/8617 strategies...
  Processed 4500/8617 strategies...
  Processed 5000/8617 strategies...
  Processed 5500/8617 strategies...
  Processed 6000/8617 strategies...
  Processed 6500/8617 strategies...
  Processed 7000/8617 strategies...
  Processed 7500/8617 strategies...
  Processed 8000/8617 strategies...
  Processed 8500/8617 strategies...

Optimization complete! Found 108 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
2,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
3,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
4,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
5,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
6,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
7,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
8,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
9,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
10,30M Trend=Aligned,622,1:3,26.5%,1.5%,1.08,0.06R,73.5%
